# Document Intelligence for Equity Derivatives

This notebook demonstrates end-to-end RAG pipelines for two document types:

| Document Type | Processing Approach | Search Strategy |
|---|---|---|
| **ISDA Legal (Master + Amendment)** | `AI_EXTRACT` → structured table + `AI_PARSE_DOCUMENT` → Cortex Search | Attribute filtering (party, doc type, governing law) |
| **Equity Research (Multimodal)** | `AI_COMPLETE` with `TO_FILE()` → text + charts | Multi-index (text + image vectors), numeric boost, time decay |

**Prerequisites:**
- ISDA PDFs and equity research PDFs pre-loaded on internal stages
- `SNOWFLAKE.CORTEX_USER` database role granted
- Warehouse available (MEDIUM recommended for AI functions)

## Setup & ISDA Structured Extraction with AI_EXTRACT

We use `AI_EXTRACT` to pull specific clauses from ISDA Master Agreements and Amendments into a relational table. This table becomes the backing store for a semantic view that the agent can query via natural language.

In [ ]:
%%sql -r isda_extract
-- ============================================================================
-- SETUP — database, schema, warehouse, stages
-- ============================================================================
CREATE DATABASE  IF NOT EXISTS CORTEX_AI_HOL;
CREATE SCHEMA    IF NOT EXISTS CORTEX_AI_HOL.RAG_PIPELINE;
USE DATABASE CORTEX_AI_HOL;
USE SCHEMA   RAG_PIPELINE;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WAREHOUSE_SIZE = 'MEDIUM' AUTO_SUSPEND = 300;
USE WAREHOUSE COMPUTE_WH;

-- Grant Cortex AI access to the active role (required for AI functions)
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE SYSADMIN;

-- Stages with server-side encryption — required for AI_EXTRACT, AI_EMBED, AI_COMPLETE
CREATE STAGE IF NOT EXISTS ISDA_DOCS
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE') DIRECTORY = (ENABLE = TRUE);
CREATE STAGE IF NOT EXISTS EQUITY_RESEARCH
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE') DIRECTORY = (ENABLE = TRUE);

-- Upload all PDFs via Snowsight  Data » Add Data » Load Files into Stage

ALTER STAGE ISDA_DOCS      REFRESH;
ALTER STAGE EQUITY_RESEARCH REFRESH;

-- Verify files are loaded before proceeding
SELECT 'ISDA' AS stage, RELATIVE_PATH, ROUND(SIZE/1024,1) AS kb
  FROM DIRECTORY(@ISDA_DOCS)
UNION ALL
SELECT 'EQUITY', RELATIVE_PATH, ROUND(SIZE/1024,1)
  FROM DIRECTORY(@EQUITY_RESEARCH)
ORDER BY 1, 2;

-- ISDA EXTRACTION: AI_EXTRACT with file-based syntax
-- =============================================================================
-- Extract structured clauses from ALL ISDA documents in the stage
CREATE OR REPLACE TABLE ISDA_AGREEMENT_TERMS AS
SELECT
    RELATIVE_PATH AS filename,
    CASE 
        WHEN RELATIVE_PATH ILIKE '%amendment%' THEN 'AMENDMENT'
        ELSE 'MASTER_AGREEMENT'
    END AS document_type,
    extraction:response:agreement_version::VARCHAR AS agreement_version,
    extraction:response:effective_date::VARCHAR AS effective_date,
    extraction:response:party_a_name::VARCHAR AS party_a,
    extraction:response:party_b_name::VARCHAR AS party_b,
    extraction:response:governing_law::VARCHAR AS governing_law,
    extraction:response:cross_default_applicable::VARCHAR AS cross_default_applicable,
    TRY_TO_NUMBER(REGEXP_REPLACE(
        extraction:response:cross_default_threshold::VARCHAR, '[^0-9.]', ''
    )) AS cross_default_threshold,
    extraction:response:cross_default_currency::VARCHAR AS cross_default_currency,
    extraction:response:automatic_early_termination_party_a::VARCHAR AS aet_party_a,
    extraction:response:automatic_early_termination_party_b::VARCHAR AS aet_party_b,
    extraction:response:closeout_method::VARCHAR AS closeout_method,
    extraction:response:netting_applicable::VARCHAR AS netting_applicable,
    extraction:response:events_of_default::VARCHAR AS events_of_default,
    extraction:response:termination_events::VARCHAR AS termination_events,
    extraction:response:payment_method::VARCHAR AS payment_method,
    extraction:response:scoring AS extraction_scores,
    extraction:response AS raw_extraction,
    CURRENT_TIMESTAMP() AS extracted_at
FROM DIRECTORY(@ISDA_DOCS),
LATERAL (
    SELECT AI_EXTRACT(
        file => TO_FILE('@ISDA_DOCS', RELATIVE_PATH),
        responseFormat => {
            'agreement_version': 'Is this a 1992 or 2002 ISDA Master Agreement? Return only 1992 or 2002',
            'effective_date': 'What is the effective date of this agreement or amendment? Return in YYYY-MM-DD format',
            'party_a_name': 'What is the full legal name of Party A (the dealer/bank)?',
            'party_b_name': 'What is the full legal name of Party B (the client/counterparty)?',
            'governing_law': 'What law governs this agreement (e.g., English law, New York law)?',
            'cross_default_applicable': 'Is the Cross Default provision applicable? Return true or false',
            'cross_default_threshold': 'What is the Cross Default Threshold Amount? Return the number and currency, or null if not specified',
            'cross_default_currency': 'What currency is the Cross Default threshold in (USD, GBP, EUR)?',
            'automatic_early_termination_party_a': 'Is Automatic Early Termination applicable to Party A? Return true or false',
            'automatic_early_termination_party_b': 'Is Automatic Early Termination applicable to Party B? Return true or false',
            'closeout_method': 'What close-out calculation method is specified for determining amounts owed upon early termination? For 2002 ISDA agreements this is typically Close-out Amount (Section 6(e)). For 1992 ISDA agreements this is typically Market Quotation or Loss. Extract the exact term used in the document; do not repeat these instructions.',
            'netting_applicable': 'Is payment netting applicable? Return true or false',
            'events_of_default': 'List all Events of Default specified in Section 5(a) as a comma-separated list',
            'termination_events': 'List all Termination Events specified in Section 5(b) as a comma-separated list',
            'payment_method': 'Is payment First Method or Second Method?'
        },
        scores => TRUE
    ) AS extraction
);

-- View results
SELECT filename, document_type, party_a, party_b, governing_law, 
       cross_default_threshold, closeout_method, extraction_scores
FROM ISDA_AGREEMENT_TERMS
ORDER BY effective_date;

In [ ]:
%%sql -r dataframe_1
SELECT filename, document_type, party_a, party_b, governing_law, 
       cross_default_threshold, closeout_method, extraction_scores
FROM ISDA_AGREEMENT_TERMS
ORDER BY effective_date;

## ISDA — Parse for RAG & Create Cortex Search Service

For full-text search over the raw document content, we use `AI_PARSE_DOCUMENT` in LAYOUT mode to preserve document structure, then create a Cortex Search Service with **attribute columns** that enable dynamic filtering at query time.

The attribute columns are key for the agent — they allow it to filter searches by party name, document type, or governing law without needing to include those constraints in the search query itself.

In [ ]:
%%sql -r isda_parse_search
CREATE OR REPLACE TABLE ISDA_PARSED_DOCUMENTS AS
    WITH parsed_docs AS (
        SELECT
            d.RELATIVE_PATH AS filename,
            CASE
                WHEN d.RELATIVE_PATH ILIKE '%amendment%' THEN 'AMENDMENT'
                ELSE 'MASTER_AGREEMENT'
            END AS document_type,
            AI_PARSE_DOCUMENT(
                TO_FILE('@ISDA_DOCS', d.RELATIVE_PATH),
                {'mode': 'LAYOUT', 'page_split': true}
            ) AS parsed
        FROM DIRECTORY(@ISDA_DOCS) d
    ),
    pages_flat AS (
        SELECT
            pd.filename,
            pd.document_type,
            ARRAY_SIZE(pd.parsed:pages)     AS page_count,
            p.value:index::INTEGER + 1      AS page_number,
            p.value:content::VARCHAR        AS page_text
        FROM parsed_docs pd,
        LATERAL FLATTEN(input => pd.parsed:pages) p
    )
    SELECT
        pf.filename,
        pf.document_type,
        t.party_a,
        t.party_b,
        t.governing_law,
        t.effective_date,
        pf.page_count,
        pf.page_number,
        pf.page_text
    FROM pages_flat pf
    LEFT JOIN ISDA_AGREEMENT_TERMS t ON t.filename = pf.filename;

SELECT *
FROM   ISDA_PARSED_DOCUMENTS
ORDER  BY filename, page_number
LIMIT  10;

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE CORTEX SEARCH SERVICE ISDA_DOCUMENT_SEARCH
      ON page_text
      ATTRIBUTES filename, document_type, party_a, party_b, governing_law,
                 effective_date, page_number, page_count
      WAREHOUSE = COMPUTE_WH
      TARGET_LAG = '1 hour'
      AS (
        SELECT
            filename,
            document_type,
            party_a,
            party_b,
            governing_law,
            effective_date,
            page_count,
            page_number,
            page_text
        FROM ISDA_PARSED_DOCUMENTS
        WHERE page_text IS NOT NULL
          AND LENGTH(page_text) > 10   
      );

## Equity Research — Multimodal Processing with AI_COMPLETE

For equity research reports that contain charts, tables, and visual elements, we use `AI_COMPLETE` with `TO_FILE()` to process PDFs directly — no parsing step needed. The model reads charts natively.

We extract both **text content** (thesis, risks, price target) and **chart/table descriptions** into a single structured output using `response_format` for guaranteed JSON.

In [ ]:
%%sql -r equity_extract
-- ============================================================================
-- EQUITY RESEARCH: AI_COMPLETE — extract ALL research reports (real + synthetic)
-- ============================================================================
-- Works for individual equity reports (ratings + price targets)
-- AND thematic/macro research (Goldman Sachs, industry surveys, etc.)
-- The enriched schema captures report_type, tickers, themes, and all visual content.
CREATE OR REPLACE TABLE EQUITY_RESEARCH_EXTRACTED AS
SELECT
    d.RELATIVE_PATH                                                         AS filename,
    -- Ticker: use AI-extracted value first, fallback to filename parsing for 2026_TICKER_* files
    COALESCE(
        NULLIF(UPPER(analysis:covered_tickers[0]::VARCHAR), ''),
        NULLIF(UPPER(SPLIT_PART(d.RELATIVE_PATH, '_', 2)), '')
    )                                                                        AS primary_ticker,
    -- Date: use AI-extracted value first, fallback to YYYY-MM-DD filename prefix
    COALESCE(
        TRY_TO_DATE(analysis:publish_date::VARCHAR),
        TRY_TO_DATE(SPLIT_PART(d.RELATIVE_PATH, '_', 1))
    )                                                                        AS publish_date,
    analysis:report_type::VARCHAR                                            AS report_type,
    analysis:primary_topic::VARCHAR                                          AS primary_topic,
    analysis:covered_tickers::VARIANT                                        AS covered_tickers,
    analysis:rating::VARCHAR                                                 AS rating,
    analysis:price_target::VARCHAR                                           AS price_target,
    analysis:investment_thesis::VARCHAR                                      AS investment_thesis,
    analysis:key_risks::VARCHAR                                              AS key_risks,
    analysis:key_themes::VARCHAR                                             AS key_themes,
    analysis:analyst_name::VARCHAR                                           AS analyst_name,
    analysis:charts::VARIANT                                                 AS charts_extracted,
    analysis:tables::VARIANT                                                 AS tables_extracted,
    -- Unified searchable text — effective for both equity and macro reports
    CONCAT(
        'Type: ',    COALESCE(analysis:report_type::VARCHAR, ''),          '. ',
        'Topic: ',   COALESCE(analysis:primary_topic::VARCHAR, ''),         '. ',
        CASE WHEN analysis:rating::VARCHAR NOT IN ('N/A','')
                 AND analysis:rating::VARCHAR IS NOT NULL
             THEN 'Rating: ' || analysis:rating::VARCHAR || '. ' ELSE '' END,
        CASE WHEN analysis:price_target::VARCHAR NOT IN ('N/A','')
                 AND analysis:price_target::VARCHAR IS NOT NULL
             THEN 'Price Target: ' || analysis:price_target::VARCHAR || '. ' ELSE '' END,
        'Themes: ',  COALESCE(analysis:key_themes::VARCHAR, ''),            '. ',
        'Thesis: ',  COALESCE(analysis:investment_thesis::VARCHAR, ''),     '. ',
        'Risks: ',   COALESCE(analysis:key_risks::VARCHAR, ''),             '. ',
        'Charts: ',  COALESCE(analysis:chart_descriptions::VARCHAR, '')
    )                                                                        AS searchable_text,
    -- Full document text from AI_PARSE_DOCUMENT — used for chunking
    full_text                                                                AS raw_text,
    analysis                                                                 AS raw_analysis,
    CURRENT_TIMESTAMP()                                                      AS processed_at
FROM DIRECTORY(@EQUITY_RESEARCH) d,
LATERAL (
    SELECT AI_COMPLETE(
        MODEL => 'claude-sonnet-4-6',
        PROMPT => PROMPT(
            'Analyze this research report. It may be individual equity research (with stock ratings and price targets) OR thematic/macro research (sector, economics, industry trends, policy). Extract ALL available information including every chart, graph, table, and visual element. For each chart: describe axes, data shown, and key insight. {0}',
            TO_FILE('@EQUITY_RESEARCH', d.RELATIVE_PATH)
        ),
        response_format => {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'report_type':       {'type': 'string',
                                          'description': 'equity_research, macro, thematic, or sector'},
                    'primary_topic':     {'type': 'string',
                                          'description': 'One sentence: what is this report about?'},
                    'covered_tickers':   {'type': 'array', 'items': {'type': 'string'},
                                          'description': 'Stock tickers explicitly covered (empty list for macro)'},
                    'rating':            {'type': 'string',
                                          'description': 'Overweight/Neutral/Underweight/Buy/Sell/N/A'},
                    'price_target':      {'type': 'string',
                                          'description': 'Price target with currency, or N/A'},
                    'investment_thesis': {'type': 'string',
                                          'description': '2-3 sentence key argument or thesis'},
                    'key_risks':         {'type': 'string',
                                          'description': 'Comma-separated key risks or downside factors'},
                    'key_themes':        {'type': 'string',
                                          'description': 'Comma-separated thematic keywords (e.g. AI, EVs, tariffs, aging)'},
                    'analyst_name':      {'type': 'string',
                                          'description': 'Lead analyst name(s)'},
                    'publish_date':      {'type': 'string',
                                          'description': 'Publication date YYYY-MM-DD; estimate from context if not explicit'},
                    'charts': {
                        'type': 'array',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'chart_title':      {'type': 'string'},
                                'chart_type':       {'type': 'string',
                                                     'description': 'bar, line, pie, scatter, heat map, waterfall, map'},
                                'data_description': {'type': 'string'},
                                'key_insight':      {'type': 'string'}
                            }
                        }
                    },
                    'tables': {
                        'type': 'array',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'table_title': {'type': 'string'},
                                'columns':     {'type': 'string'},
                                'summary':     {'type': 'string'}
                            }
                        }
                    },
                    'chart_descriptions': {
                        'type': 'string',
                        'description': 'Detailed natural-language description of ALL charts, graphs, and visual elements for semantic search indexing'
                    }
                },
                'required': ['report_type','primary_topic','covered_tickers','rating',
                             'price_target','investment_thesis','key_risks','key_themes',
                             'analyst_name','publish_date','charts','tables','chart_descriptions']
            }
        }
    ) AS analysis,
        AI_PARSE_DOCUMENT(
            TO_FILE('@EQUITY_RESEARCH', d.RELATIVE_PATH),
            {'mode': 'LAYOUT'}
        ):content::VARCHAR AS full_text
)
WHERE d.RELATIVE_PATH LIKE '%.pdf';

-- Preview results — equity reports show ratings, macro reports show topics
SELECT filename, report_type, primary_ticker, rating, price_target,
       ARRAY_SIZE(charts_extracted) AS num_charts,
       ARRAY_SIZE(tables_extracted) AS num_tables
FROM   EQUITY_RESEARCH_EXTRACTED
ORDER  BY report_type, filename;

In [ ]:
%%sql -r dataframe_4
USE DATABASE CORTEX_AI_HOL;

USE SCHEMA RAG_PIPELINE;

-- Preview extraction results
SELECT
    filename,
    primary_ticker,
    rating,
    price_target,
    analyst_name,
    ARRAY_SIZE(charts_extracted) AS num_charts,
    ARRAY_SIZE(tables_extracted) AS num_tables,
    charts_extracted,
    tables_extracted
FROM
    EQUITY_RESEARCH_EXTRACTED
ORDER BY
    publish_date;

### Synthetic Readership Table

In production this data would come from your document management system.
We create synthetic `view_count` values to demonstrate **numeric boosting** in Cortex Search:
popular articles (high view count) rank higher at query time when boosting is enabled.

In [ ]:
%%sql -r equity_readership
-- Synthetic readership data for numeric boost demonstration.
-- Real institutional research gets higher baseline readership than synthetic files.
CREATE OR REPLACE TABLE EQUITY_RESEARCH_READERSHIP AS
SELECT
    filename,
    primary_ticker,
    publish_date,
    report_type,
    UNIFORM(500, 5000, RANDOM())
        + CASE WHEN filename NOT LIKE '2026%' THEN 8000 ELSE 0 END
        + CASE WHEN primary_ticker IN ('NVDA','AAPL','MSFT','GOOGL') THEN 3000 ELSE 0 END
        + CASE WHEN publish_date > DATEADD('month', -3, CURRENT_DATE()) THEN 2000 ELSE 0 END
    AS view_count,
    UNIFORM(5, 10, RANDOM())    AS analyst_reputation_score
FROM EQUITY_RESEARCH_EXTRACTED;

SELECT filename, primary_ticker, report_type, view_count
FROM   EQUITY_RESEARCH_READERSHIP
ORDER  BY view_count DESC;

### Chunk Raw Text for Cortex Search

Large reports (Goldman Sachs, G10FX) are hundreds of pages — one row per document
would make Cortex Search return huge blobs. `SPLIT_TEXT_RECURSIVE_CHARACTER` returns
an **array**, so we `LATERAL FLATTEN` it into one row per chunk.
Each chunk gets the report metadata prepended so it's self-contained for retrieval.

In [ ]:
%%sql -r equity_chunks
-- =============================================================================
-- CHUNK RAW TEXT FOR CORTEX SEARCH
-- =============================================================================
-- SPLIT_TEXT_RECURSIVE_CHARACTER returns an ARRAY — use LATERAL FLATTEN,
-- NOT TABLE().  Each chunk gets report metadata prepended so every chunk
-- is self-contained for retrieval without needing a JOIN at query time.
CREATE OR REPLACE TABLE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_CHUNKS AS
SELECT
    e.filename,
    e.primary_ticker,
    e.publish_date,
    e.report_type,
    e.analyst_name,
    e.rating,
    f.index                 AS chunk_index,
    f.value::VARCHAR        AS chunk_text,
    -- Prepend metadata so every chunk is self-contained for retrieval
    CONCAT(
        'Report: ', e.filename,       ' | ',
        'Type: ',   COALESCE(e.report_type, ''),   ' | ',
        'Topic: ',  COALESCE(e.primary_topic, ''),  ' | ',
        CASE WHEN e.rating NOT IN ('N/A','') AND e.rating IS NOT NULL
             THEN 'Rating: ' || e.rating || ' | ' ELSE '' END,
        f.value::VARCHAR
    )                       AS searchable_chunk
FROM CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_EXTRACTED e,
LATERAL FLATTEN(
    input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
        e.raw_text,
        'none',   -- language (none = language-agnostic splitting)
        1500,     -- chunk size in characters
        150       -- overlap to preserve context across boundaries
    )
) f
WHERE e.raw_text IS NOT NULL
  AND LENGTH(e.raw_text) > 0;

-- Verify: large reports should produce many chunks
SELECT report_type,
       COUNT(*)                      AS total_chunks,
       COUNT(DISTINCT filename)       AS reports,
       ROUND(AVG(LENGTH(chunk_text))) AS avg_chunk_chars
FROM   CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_CHUNKS
GROUP  BY 1
ORDER  BY 1;

## Cell 4: Multimodal Cortex Search — voyage-multimodal-3 Image Embeddings

**True multimodal RAG**: we render each equity research PDF page as a PNG image,
embed it with `voyage-multimodal-3`, and store those vectors in the Cortex Search Service.

Why does this matter? `voyage-multimodal-3` maps text **and** images into the *same*
vector space. A text query like `"revenue growth bar charts"` can retrieve visually
similar chart pages — even if the words don't appear in the image.

```
┌───────────────────────────────────────────────────────────┐
│            EQUITY_RESEARCH_SEARCH                          │
├─────────────────────────┬─────────────────────────────────┤
│  TEXT INDEX             │  VECTOR INDEX                   │
│  (searchable_text)      │  (image_embedding)              │
│  keyword + semantic     │  voyage-multimodal-3 image vec  │
│                         │  1 row per PDF page             │
├─────────────────────────┴─────────────────────────────────┤
│  ATTRIBUTES: ticker, rating, publish_date,                 │
│              analyst_name, view_count                      │
└───────────────────────────────────────────────────────────┘
```

**Query flow**: embed a text string with `voyage-multimodal-3` → pass as
`multi_index_query.image_embedding.vector` alongside the text query.

In [ ]:
# Install PyMuPDF for PDF→PNG rendering
# ── PyPI access prerequisite ──────────────────────────────────────────────
# !pip install requires either:
#   A) Artifact Repository: set 'snowflake.snowpark.pypi_shared_repository'
#      in the notebook Edit Service dialog (recommended — no setup needed)
#   B) EAI: run the pypi_access EAI setup in 02_agent_and_evaluation Step 4a
# ─────────────────────────────────────────────────────────────────────────
!pip install pymupdf --quiet

from snowflake.snowpark.context import get_active_session
import fitz, requests, os

session = get_active_session()

session.sql("""
    CREATE STAGE IF NOT EXISTS CORTEX_AI_HOL.RAG_PIPELINE.CHART_IMAGES
      ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE') DIRECTORY = (ENABLE = TRUE)
""").collect()

# Process ALL PDFs — synthetic (2026*) and real GS reports
# Render first 5 pages of large reports to keep costs reasonable
pdfs = session.sql("""
    SELECT RELATIVE_PATH, SIZE
    FROM   DIRECTORY(@CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH)
    WHERE  RELATIVE_PATH LIKE '%.pdf'
    ORDER  BY RELATIVE_PATH
""").to_pandas()

print(f'Rendering {len(pdfs)} PDFs to PNG images...')
total = 0

for _, row in pdfs.iterrows():
    pdf_path = row['RELATIVE_PATH']
    size_mb  = row['SIZE'] / 1024 / 1024
    base     = os.path.splitext(pdf_path)[0]
    max_pages = 99 if size_mb < 0.1 else 5   # cap large real reports at 5 pages

    url = session.sql(f"""
        SELECT GET_PRESIGNED_URL(@CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH,
                                '{pdf_path}', 3600) AS url
    """).collect()[0]['URL']

    pdf_bytes = requests.get(url, timeout=60).content
    doc       = fitz.open(stream=pdf_bytes, filetype='pdf')

    for page_idx in range(min(len(doc), max_pages)):
        pix      = doc[page_idx].get_pixmap(matrix=fitz.Matrix(2, 2))
        img_name = f'{base}_p{page_idx + 1:02d}.png'
        tmp_path = f'/tmp/{img_name}'
        pix.save(tmp_path)
        session.file.put(tmp_path, '@CORTEX_AI_HOL.RAG_PIPELINE.CHART_IMAGES',
                         auto_compress=False, overwrite=True)
        os.remove(tmp_path)
        total += 1

    print(f'  {len(doc):2d}p (rendered {min(len(doc),max_pages)}) {pdf_path}')
    doc.close()

session.sql('ALTER STAGE CORTEX_AI_HOL.RAG_PIPELINE.CHART_IMAGES REFRESH').collect()
print(f'\n✓ {total} page images uploaded to @CHART_IMAGES')

session.sql("""
    SELECT RELATIVE_PATH, ROUND(SIZE/1024,0) AS kb
    FROM   DIRECTORY(@CORTEX_AI_HOL.RAG_PIPELINE.CHART_IMAGES)
    ORDER  BY RELATIVE_PATH
""").to_pandas()

In [ ]:
%%sql -r image_embeddings
-- =============================================================================
-- EMBED CHART IMAGES WITH voyage-multimodal-3
-- =============================================================================
-- voyage-multimodal-3 creates 1024-dim vectors that align text and image
-- representations in the same embedding space.
CREATE OR REPLACE TABLE CORTEX_AI_HOL.RAG_PIPELINE.CHART_IMAGE_EMBEDDINGS AS
SELECT
    RELATIVE_PATH                                                           AS image_path,
    -- Reconstruct the PDF filename: strip _pNN.png suffix, add .pdf
    REGEXP_REPLACE(RELATIVE_PATH, '_p\\d+\\.png$', '.pdf')                  AS report_filename,
    REGEXP_REPLACE(RELATIVE_PATH, '^.*_p(\\d+)\\.png$', '\\1')::INTEGER      AS page_num,
    AI_EMBED('voyage-multimodal-3',
             TO_FILE('@CORTEX_AI_HOL.RAG_PIPELINE.CHART_IMAGES', RELATIVE_PATH)
    )                                                                        AS image_embedding
FROM DIRECTORY(@CORTEX_AI_HOL.RAG_PIPELINE.CHART_IMAGES)
WHERE RELATIVE_PATH LIKE '%.png';

-- Preview: confirm embeddings were created (dimension = 1024 for voyage-multimodal-3)
SELECT
    image_path,
    report_filename,
    page_num,
    ARRAY_SIZE(image_embedding::ARRAY) AS embedding_dims
FROM   CORTEX_AI_HOL.RAG_PIPELINE.CHART_IMAGE_EMBEDDINGS
ORDER  BY image_path;

In [ ]:
 CREATE OR REPLACE TABLE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH_BASE AS
    SELECT
        i.image_path,
        i.report_filename                                     AS filename,
        i.page_num,
        c.chunk_index,
        c.primary_ticker,
        c.rating,
        c.analyst_name,
        c.publish_date,
        r.view_count,
        c.searchable_chunk,
        i.image_embedding
    FROM CORTEX_AI_HOL.RAG_PIPELINE.CHART_IMAGE_EMBEDDINGS    i
    JOIN CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_CHUNKS    c
      ON  i.report_filename = c.filename
    JOIN CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_READERSHIP r
      ON  c.filename = r.filename;


In [ ]:
%%sql -r create_equity_search
-- =============================================================================
-- CREATE MULTI-INDEX CORTEX SEARCH SERVICE
-- =============================================================================
-- TEXT INDEXES  → searchable_chunk (managed arctic embeddings)
-- VECTOR INDEXES → image_embedding (pre-computed voyage-multimodal-3)
CREATE OR REPLACE CORTEX SEARCH SERVICE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH
  TEXT INDEXES (searchable_chunk)
  VECTOR INDEXES (image_embedding)
  ATTRIBUTES primary_ticker, rating, analyst_name, publish_date, view_count, page_num, chunk_index
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 hour'
  AS (
    SELECT
        image_path,
        filename,
        page_num,
        chunk_index,
        primary_ticker,
        rating,
        analyst_name,
        publish_date,
        view_count,
        searchable_chunk,
        image_embedding
    FROM CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH_BASE
  );

SHOW CORTEX SEARCH SERVICES IN SCHEMA CORTEX_AI_HOL.RAG_PIPELINE;

### Voyage Base Table + Agent-Friendly Search Service

The voyage-multimodal-3 CSS above (page-level image vectors) powers the **stored procedure** custom tool in Part 2.

We also build `EQUITY_RESEARCH_SEARCH_AGENT` with managed embeddings so the agent can issue plain `query` searches — useful for direct tool calls without a procedure.

In [ ]:
%%sql -r equity_voyage_base
-- ============================================================================
-- VOYAGE EMBEDDING BASE — one row per report, voyage-multimodal-3 text vectors
-- ============================================================================
-- voyage-multimodal-3 maps text AND images into the same vector space.
-- Embedding text with this model lets text queries find visually similar content.
CREATE OR REPLACE TABLE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_VOYAGE_BASE AS
SELECT
    e.filename,
    e.primary_ticker,
    e.report_type,
    e.rating,
    e.price_target,
    e.analyst_name,
    r.publish_date,
    r.view_count,
    e.searchable_text,
    COALESCE(e.raw_analysis:chart_descriptions::VARCHAR, '') AS chart_content,
    -- voyage-multimodal-3: text embedding aligned with image vector space
    AI_EMBED('voyage-multimodal-3', e.searchable_text)       AS voyage_embedding
FROM CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_EXTRACTED  e
JOIN CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_READERSHIP r
  ON e.filename = r.filename;

SELECT filename, primary_ticker, report_type,
       ARRAY_SIZE(voyage_embedding::ARRAY) AS voyage_dims
FROM   CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_VOYAGE_BASE
ORDER  BY filename;

In [ ]:
%%sql -r create_agent_search
  CREATE OR REPLACE CORTEX SEARCH SERVICE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH_AGENT
      ON searchable_chunk
      ATTRIBUTES primary_ticker, report_type, rating, analyst_name,
                 publish_date, view_count, filename, chunk_index
      WAREHOUSE = COMPUTE_WH
      TARGET_LAG = '1 hour'
      AS (
        SELECT
            c.filename,
            c.primary_ticker,
            c.report_type,
            c.rating,
            c.analyst_name,
            c.publish_date,
            r.view_count,
            c.chunk_index,
            c.chunk_text,
            c.searchable_chunk
        FROM CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_CHUNKS      c
        JOIN CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_READERSHIP  r
          ON c.filename = r.filename
      );

    GRANT USAGE ON CORTEX SEARCH SERVICE CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH_AGENT
      TO ROLE PUBLIC;

### Multimodal Query Demos

With user-provided `VECTOR INDEXES` you **must** use `multi_index_query` instead of `query`.
The trick: `AI_EMBED('voyage-multimodal-3', 'your text')` returns a vector in the **same
space** as the image embeddings — so text queries find visually similar chart pages.

| Query key | Value | Which index |
|---|---|---|
| `searchable_text` | `{"query": "..."}` | Text index — keyword + semantic |
| `image_embedding` | `{"vector": [...]}` | Vector index — visual similarity |

In [ ]:
# Demo 1: Multimodal query — text + image vectors, numeric boost + time decay
# ─────────────────────────────────────────────────────────────────────────────
# SEARCH_PREVIEW requires a constant string as its 2nd argument, so we cannot
# pass TO_JSON(OBJECT_CONSTRUCT(...vec...)) directly in SQL.
# Pattern: embed the query in Python → build JSON payload → pass as literal.
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

query = 'NVIDIA competitive risks and market share threats'

# Step 1 — compute voyage-multimodal-3 embedding (same space as page images)
# ARRAY columns return as a JSON string in Snowpark — parse with json.loads
vec = json.loads(
    session.sql(
        f"SELECT AI_EMBED('voyage-multimodal-3', '{query}')::ARRAY AS v"
    ).collect()[0]['V']
)

# Step 2 — build JSON payload
payload = json.dumps({
    'multi_index_query': {
        'searchable_chunk': {'text': 'NVIDIA competitive risks'},
        'image_embedding':  {'vector': vec}
    },
    'columns': ['image_path', 'filename', 'page_num', 'chunk_index',
                'primary_ticker', 'rating', 'analyst_name',
                'publish_date', 'view_count', 'searchable_chunk'],
    'limit': 5,
    'boosts': {'view_count': {'boost_by': 'multiplier'}},
    'decays': {'publish_date': {'decay_speed': 'fast'}}
})

# Step 3 — call SEARCH_PREVIEW with the payload as a constant string
results = session.sql(f"""
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH',
            $${payload}$$
        )
    ) AS results
""").collect()[0]['RESULTS']

print(json.dumps(json.loads(results), indent=2))


In [ ]:
# Demo 2: Visual-heavy query — weight image vector more than text
# Higher vector_weight surfaces pages whose visual layout matches the query
# even when the exact words don't appear in the chunk text.
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

query = 'earnings revenue growth bar chart technology sector'

# ARRAY columns return as a JSON string in Snowpark — parse with json.loads
vec = json.loads(
    session.sql(
        f"SELECT AI_EMBED('voyage-multimodal-3', '{query}')::ARRAY AS v"
    ).collect()[0]['V']
)

payload = json.dumps({
    'multi_index_query': {
        'searchable_chunk': {'text': 'earnings revenue growth chart'},
        'image_embedding':  {'vector': vec}
    },
    'columns': ['image_path', 'filename', 'page_num', 'chunk_index',
                'primary_ticker', 'rating', 'analyst_name',
                'publish_date', 'view_count', 'searchable_chunk'],
    'limit': 5,
    'scoring': {
        'component_weights': {'text_weight': 0.2, 'vector_weight': 0.8}
    },
    'boosts': {'view_count': {'boost_by': 'multiplier'}}
})

results = session.sql(f"""
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH',
            $${payload}$$
        )
    ) AS results
""").collect()[0]['RESULTS']

print(json.dumps(json.loads(results), indent=2))

In [ ]:
# Demo 3: Filtered multimodal search — Overweight-rated stocks only, time decay
# @eq filter on the rating ATTRIBUTE applies before scoring —
# only rows matching the filter are ranked, reducing noise.
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

query = 'strong earnings growth positive outlook buy'

# ARRAY columns return as a JSON string in Snowpark — parse with json.loads
vec = json.loads(
    session.sql(
        f"SELECT AI_EMBED('voyage-multimodal-3', '{query}')::ARRAY AS v"
    ).collect()[0]['V']
)

payload = json.dumps({
    'multi_index_query': {
        'searchable_chunk': {'text': 'strong earnings growth positive outlook'},
        'image_embedding':  {'vector': vec}
    },
    'filter':  {'@eq': {'rating': 'Overweight'}},
    'columns': ['image_path', 'filename', 'page_num', 'chunk_index',
                'primary_ticker', 'rating', 'analyst_name',
                'publish_date', 'view_count', 'searchable_chunk'],
    'limit': 5,
    'decays': {'publish_date': {'decay_speed': 'moderate'}}
})

results = session.sql(f"""
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'CORTEX_AI_HOL.RAG_PIPELINE.EQUITY_RESEARCH_SEARCH',
            $${payload}$$
        )
    ) AS results
""").collect()[0]['RESULTS']

print(json.dumps(json.loads(results), indent=2))

## Summary

### What we built:

| Component | Purpose | Key Feature |
|---|---|---|
| `ISDA_AGREEMENT_TERMS` table | Structured clause data from AI_EXTRACT | Extraction scores for confidence |
| `ISDA_DOCUMENT_SEARCH` CSS | Full-text RAG over ISDA docs | Attribute filtering (party, doc type, law) |
| `EQUITY_RESEARCH_EXTRACTED` table | Multimodal extraction (text + charts) | JSON schema guarantees structure |
| `EQUITY_RESEARCH_SEARCH` CSS | Multi-index text + image search | Numeric boost, time decay, component weights |

### Cortex Search Scoring Options Demonstrated:

1. **Numeric Boosts** — `view_count` multiplier to surface popular research
2. **Time Decays** — `publish_date` decay to prioritize recent content
3. **Component Weights** — Adjust `text_weight` vs `vector_weight` ratio per query type
4. **Attribute Filtering** — `@eq`, `@in` operators on metadata columns

### Next Steps:
- **Create semantic view** on `ISDA_AGREEMENT_TERMS` via the Snowsight UI (see lab guide)
- **Install marketplace data** for structured financial data
- **Build the Cortex Agent** that combines all these tools (Part 2)